# 04 · Modelo de estancia y factores de demora (LightGBM + SHAP)
- **Modelo al ingreso**: solo información disponible en las primeras 2 h → útil para priorizar.
- **Modelo explicativo**: toda la información del episodio → para entender qué alarga la estancia.
- Objetivos: *estancia prolongada* en urgencias (> P75) y *necesidad de hospitalización*.
- Validación **temporal**: entrena con el pasado, evalúa con los últimos meses.

In [ ]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from src import datos, flow
from src.config import FIGURES, DATA_INTERIM, DATA_PROCESSED, ETAPAS
pd.set_option("display.max_columns", 50); pd.set_option("display.float_format", "{:,.2f}".format)
sns.set_theme(style="whitegrid", context="notebook")
def guardar(nombre):
    plt.tight_layout(); plt.savefig(FIGURES / f"{nombre}.png", dpi=150, bbox_inches="tight"); plt.show()
from src import models
from src.datos import FEATURES_INGRESO, FEATURES_EXPLICATIVO
import shap

In [ ]:
ep = datos.episodios()
ep = ep[(ep.primera_etapa != 9201) & (ep.los_atipica == 0)].copy()   # entran por urgencias, sin estancias atípicas
urg = ep[(~ep.hospitalizado) & (ep.atencion_breve == 0)].copy()   # urgencias con observación (>= 15 min)
p75 = urg.los_h.quantile(.75); urg["prolongada"] = (urg.los_h > p75).astype(int)
print(f"Episodios: {len(ep):,} · Urgencias con observación: {len(urg):,} · umbral estancia prolongada P75 = {p75:.1f} h")

## A. Estancia prolongada en urgencias (al ingreso)

In [ ]:
tr, te, cut = models.temporal_split(urg, "inicio")
X = models.prepare_X(urg, FEATURES_INGRESO); y = urg.prolongada
m_prol = models.train_classifier(X.loc[tr], y.loc[tr], X.loc[te], y.loc[te])
print("corte temporal:", cut); models.eval_classification(m_prol, X.loc[te], y.loc[te])

In [ ]:
Xr = models.prepare_X(urg, FEATURES_INGRESO)
m_reg = models.train_regressor(Xr.loc[tr], urg.log_los.loc[tr], Xr.loc[te], urg.log_los.loc[te])
models.eval_regression_log(m_reg, Xr.loc[te], urg.log_los.loc[te])

In [ ]:
_, sv, Xs = models.shap_values(m_prol, X.loc[te], 5000)
shap.plots.beeswarm(sv[:, :, 1] if sv.values.ndim == 3 else sv, max_display=15, show=False); guardar("04_shap_prolongada_ingreso")
imp_prol = models.shap_importance_table(sv, Xs); imp_prol.head(15)

## B. Estancia prolongada (explicativo, con toda la información del episodio)

In [ ]:
Xe = models.prepare_X(urg, FEATURES_EXPLICATIVO)
m_exp = models.train_classifier(Xe.loc[tr], y.loc[tr], Xe.loc[te], y.loc[te])
print(models.eval_classification(m_exp, Xe.loc[te], y.loc[te]))
_, sv_e, Xs_e = models.shap_values(m_exp, Xe.loc[te], 5000)
shap.plots.bar(sv_e[:, :, 1] if sv_e.values.ndim == 3 else sv_e, max_display=15, show=False); guardar("04_shap_prolongada_explicativo")
imp_exp = models.shap_importance_table(sv_e, Xs_e); imp_exp.head(15)

In [ ]:
for v in ["ocupacion_urg_llegada", "n_labs", "hora_llegada", "edad"]:
    if v in Xs_e:
        vals = sv_e[:, :, 1] if sv_e.values.ndim == 3 else sv_e
        shap.plots.scatter(vals[:, v], show=False); guardar(f"04_shap_dep_{v}")

## C. ¿Quién terminará hospitalizado? (al ingreso)

In [ ]:
ep["y_hosp"] = ep.hospitalizado.astype(int)
tr2, te2, _ = models.temporal_split(ep, "inicio")
Xh = models.prepare_X(ep, FEATURES_INGRESO)
m_hosp = models.train_classifier(Xh.loc[tr2], ep.y_hosp.loc[tr2], Xh.loc[te2], ep.y_hosp.loc[te2])
print(models.eval_classification(m_hosp, Xh.loc[te2], ep.y_hosp.loc[te2]))
_, sv_h, Xs_h = models.shap_values(m_hosp, Xh.loc[te2], 5000)
imp_hosp = models.shap_importance_table(sv_h, Xs_h); imp_hosp.head(10)

### Perfiles: diagnósticos con mayor estancia y hospitalización

In [ ]:
dx = ep.groupby("dx_principal_nombre").agg(episodios=("episodio_id", "size"), estancia_p50_h=("los_h", "median"),
    pct_hospitalizado=("hospitalizado", "mean"), pct_reconsulta_72h=("reconsulta_72h", "mean"))
dx = dx[dx.episodios >= 500].sort_values("episodios", ascending=False); dx["pct_hospitalizado"] *= 100; dx["pct_reconsulta_72h"] *= 100
dx.head(20)

In [ ]:
import joblib
from src.config import MODELS
joblib.dump({"prolongada": m_prol, "explicativo": m_exp, "hospitalizacion": m_hosp}, MODELS / "modelos_estancia.joblib")
pd.concat([imp_prol.assign(modelo="prolongada_ingreso"), imp_exp.assign(modelo="prolongada_explicativo"),
           imp_hosp.assign(modelo="hospitalizacion_ingreso")]).to_parquet(DATA_INTERIM / "shap_importancia.parquet", index=False)
dx.reset_index().to_parquet(DATA_INTERIM / "perfil_dx.parquet", index=False)